*0.1 Python for GenAI*

# requests

**The situation.** A nightly report job built years ago on the `requests` library hangs at 3 a.m. and is still hanging at 9. The provider had a short network problem. The bare `requests.post()` had no time limit, so it waited forever, and the job's lock stopped every later run.

**The fix: the same call, made safe.** `requests` is the older library for web requests and you will meet it in existing code. It needs three things it does not do by itself: a `Session` (reuses connections), a retry adapter (backoff on 429 and 5xx), and — the one that would have saved the 3 a.m. job — a `timeout` on every call.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import os

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

session = requests.Session()
session.mount(
    "https://",
    HTTPAdapter(
        max_retries=Retry(
            total=3,
            backoff_factor=0.5,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["POST"],
        )
    ),
)
session.headers["Authorization"] = f"Bearer {os.environ['OPENAI_API_KEY']}"
body = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "Reply with the single word: pong"}],
    "max_tokens": 3,
}

response = session.post(
    "https://api.openai.com/v1/chat/completions", json=body, timeout=(5, 30)
)  # (connect, read)
response.raise_for_status()
print(
    "status:", response.status_code, "| reply:", response.json()["choices"][0]["message"]["content"]
)
assert response.ok

status: 200 | reply: Pong


**Reading the output.** Same call, same reply as httpx. The difference is invisible until something goes wrong — so make something go wrong.

**The 3 a.m. bug, reproduced safely.** A timeout of one millisecond cannot succeed. With no timeout this would hang; with one, it fails immediately and says so.

In [3]:
try:
    session.post("https://api.openai.com/v1/chat/completions", json=body, timeout=0.001)
except requests.exceptions.RequestException as error:
    print("timeout=0.001 →", type(error).__name__, "— the job fails fast instead of hanging")
    timeout_error = type(error).__name__
assert "Timeout" in timeout_error or "Connect" in timeout_error

timeout=0.001 → ConnectTimeout — the job fails fast instead of hanging


**The rule to remember.** With `requests`, `timeout=` on every single call is not optional. There is no default.

| Use it when | Don't when | Instead use |
|---|---|---|
| maintaining existing code that uses it | new async services | httpx |

**Watch out**
- Put the retry adapter on the session so every call gets it; do not hand-write retry loops.
- Retrying a POST can repeat a side effect; the adapter retries POST only because `allowed_methods` says so — make sure the server can handle a duplicate.
- Keep `requests` for code you inherit; write new services with httpx.